# 🍛 Train YOLO11 Medium on Merged Indian Food Dataset (15,000 Images)
This notebook automatically downloads **IndianFoodNet** and **Indian_food-2**, merges them into a unified 31-class dataset, and trains **YOLO11 Medium** with smashed/mushy food waste augmentations.

In [ ]:
# 1. Install dependencies
!pip install -q ultralytics roboflow pyyaml

In [ ]:
# 2. Download Datasets & Merge in Kaggle Cloud
import os
import shutil
import yaml
from pathlib import Path
from roboflow import Roboflow

print("Step 1/3: Downloading IndianFoodNet (13,000+ images)...")
rf1 = Roboflow(api_key="AaRnYG0DitwWYxSXdHso")
ds1 = rf1.workspace("indianfoodnet").project("indianfoodnet").version(1).download("yolov8")

print("Step 2/3: Downloading Indian_food-2 (1,950 images)...")
rf2 = Roboflow(api_key="WhkO46oGArsxkjHLp95p")
ds2 = rf2.workspace("indianfood").project("indian_food-pwzlc").version(2).download("yolov8")

print("Step 3/3: Merging datasets into unified master dataset...")
DATASET_A = Path(ds1.location)
DATASET_B = Path(ds2.location)
MERGED_DIR = Path("/kaggle/working/merged_indian_food")

SYNONYMS = {
    "besan_cheela": "BesanCheela",
    "dosa": "Dosa",
    "gulab_jamun": "GulabJamun",
    "idli": "Idli",
    "palak_paneer": "PalakPaneer",
    "poha": "Poha",
    "samosa": "Samosa",
}

def load_classes(yaml_path):
    with open(yaml_path, "r") as f:
        return yaml.safe_load(f)["names"]

def normalize_name(name):
    return SYNONYMS.get(name.strip(), name.strip())

names_a = load_classes(DATASET_A / "data.yaml")
names_b = load_classes(DATASET_B / "data.yaml")

master_classes = sorted(list(set([normalize_name(n) for n in names_a + names_b])))
class_to_id = {c: i for i, c in enumerate(master_classes)}

for split in ["train", "valid", "test"]:
    (MERGED_DIR / split / "images").mkdir(parents=True, exist_ok=True)
    (MERGED_DIR / split / "labels").mkdir(parents=True, exist_ok=True)

def process_dataset(ds_path, prefix, ds_names):
    id_remap = {i: class_to_id[normalize_name(n)] for i, n in enumerate(ds_names)}
    for split in ["train", "valid", "test"]:
        img_dir = ds_path / split / "images"
        lbl_dir = ds_path / split / "labels"
        if not img_dir.exists():
            continue
        
        target_img_dir = MERGED_DIR / split / "images"
        target_lbl_dir = MERGED_DIR / split / "labels"
        
        for img_file in img_dir.glob("*.*"):
            stem = img_file.stem
            new_stem = f"{prefix}_{stem}"
            target_img = target_img_dir / f"{new_stem}{img_file.suffix}"
            if not target_img.exists():
                os.link(img_file, target_img)
                
            lbl_file = lbl_dir / f"{stem}.txt"
            if lbl_file.exists():
                target_lbl = target_lbl_dir / f"{new_stem}.txt"
                with open(lbl_file, "r") as f_in, open(target_lbl, "w") as f_out:
                    for line in f_in:
                        parts = line.strip().split()
                        if len(parts) >= 5:
                            try:
                                old_cls = int(parts[0])
                                print(id_remap[old_cls], *parts[1:], file=f_out)
                            except (ValueError, KeyError):
                                continue

process_dataset(DATASET_A, "ifn", names_a)
process_dataset(DATASET_B, "if2", names_b)

data_yaml = {
    "path": str(MERGED_DIR),
    "train": "train/images",
    "val": "valid/images",
    "test": "test/images",
    "nc": len(master_classes),
    "names": master_classes
}

with open(MERGED_DIR / "data.yaml", "w") as f:
    yaml.dump(data_yaml, f, sort_keys=False)

train_count = len(list((MERGED_DIR / "train" / "images").glob("*.*")))
val_count = len(list((MERGED_DIR / "valid" / "images").glob("*.*")))
print(f"Merge complete! Total: {train_count + val_count} images across {len(master_classes)} classes.")

In [ ]:
# 3. Train YOLO11 Medium with Checkpointing
import os
from ultralytics import YOLO

merged_yaml = "/kaggle/working/merged_indian_food/data.yaml"
checkpoint_path = "/kaggle/working/runs/indian_food_yolo/weights/last.pt"

if os.path.exists(checkpoint_path):
    print(f"Checkpoint detected! Resuming from: {checkpoint_path}")
    model = YOLO(checkpoint_path)
    results = model.train(resume=True)
else:
    print("Starting fresh training on Dual T4 GPUs with YOLO11 Medium...")
    model = YOLO("yolo11m.pt")
    results = model.train(
        data=merged_yaml,
        epochs=30,
        imgsz=640,
        batch=32,
        device=[0, 1],
        workers=4,
        save=True,
        save_period=5,
        mosaic=1.0,
        mixup=0.25,
        degrees=15.0,
        shear=5.0,
        hsv_s=0.7,
        hsv_v=0.4,
        name="indian_food_yolo",
        project="/kaggle/working/runs"
    )

print("Training complete!")

In [ ]:
# 4. Zip the weights for easy 1-click download
import shutil
weights_dir = "/kaggle/working/runs/indian_food_yolo/weights"
shutil.make_archive("/kaggle/working/yolo26_weights", "zip", weights_dir)
print("Weights zipped! Download 'yolo26_weights.zip' from the Output tab.")